In [76]:
# get judgement of each question. 
import json
# folder = "../inference/output/GLM-4.6/browsecomp/20251105-171414"
folder = "/mnt/sharefs/users/hao.zhang/ds8-agent/OSDI2025/DeepResearch/inference/output/GLM-4.6/browsecomp/20251105-214334"
iteration_file = "iter3_scored.jsonl"
file_path = f"{folder}/{iteration_file}"

def get_iteration_judgement(file_path):
    verdicts = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            data = json.loads(line) 
            if "is_correct" in data.keys():
                verdict = "correct" if data['is_correct'] else "incorrect"
            else:
                verdict = "unknown"
                print(f" ERROR: NO verdict found...")
            verdicts.append(verdict)
    return verdicts

for v in get_iteration_judgement(file_path):
    print(v)
    

    

correct
correct
correct
incorrect
correct
incorrect
correct
incorrect
correct
incorrect
incorrect
correct
incorrect
incorrect
incorrect
incorrect
incorrect
correct
correct
correct
incorrect
incorrect
incorrect
incorrect
incorrect
correct
incorrect
incorrect
correct
correct


In [ ]:
# sort questions based on order in DS 
# load dataset 
# get first 30. with idx 
import os 
import json


dataset = "browsecomp"
debug_size=30
data_filepath = os.path.join("..", "inference", "eval_data", f"{dataset}.jsonl")
try:
    if data_filepath.endswith(".json"):
        with open(data_filepath, "r", encoding="utf-8") as f:
            items = json.load(f)
        if not isinstance(items, list):
            raise ValueError("Input JSON must be a list of objects.")
        if items and not isinstance(items[0], dict):
            raise ValueError("Input JSON list items must be objects.")
    elif data_filepath.endswith(".jsonl"):
        with open(data_filepath, "r", encoding="utf-8") as f:
            items = [json.loads(line) for line in f]
    else:
        raise ValueError("Unsupported file extension. Please use .json or .jsonl files.")
    items = items
except FileNotFoundError:
    print(f"Error: Input file not found at {data_filepath}")
    exit(1)
except (json.JSONDecodeError, ValueError) as e:
    print(f"Error reading or parsing input file {data_filepath}: {e}")
    exit(1)

items = items[:debug_size]
print(len(items))
print(items[0]["question"])
###########################################################################
# get a input file, 
folder =  "../inference/output/GLM-4.6/browsecomp/20251105-214334"
iteration_file = "iter1.evolved_kflow.jsonl"
file_path = f"{folder}/{iteration_file}"
# Extract input data 
data_lines = []
with open(file_path, 'r') as f:
    for i, line in enumerate(f, start=1):
        data = json.loads(line)
        data_lines.append(data)

# Reorder data_lines based on the order in items["question"]
question_order = {item["question"]: idx for idx, item in enumerate(items)}
sorted_data_lines = sorted(data_lines, key=lambda x: question_order.get(x["question"], float('inf')))

print(f"Original data_lines count: {len(data_lines)}")
print(f"Sorted data_lines count: {len(sorted_data_lines)}")

# verify.
for i in range(len(data_lines)):
    print(f"{i}:{data_lines[i]['question'][:20]}")
    assert items[i]['question'][:20] == sorted_data_lines[i]['question'][:20]

########################################################################
# Output new file - replace .jsonl with .sorted.jsonl
# output_file = file_path.replace(".jsonl", ".sorted.jsonl")
output_file = file_path.replace(".jsonl", ".jsonl")
with open(output_file, 'w') as f:
    for data in sorted_data_lines:
        f.write(json.dumps(data) + '\n')

print(f"Sorted data written to: {output_file}")


In [83]:
# Get reflector analysis and agreement with ground truth
import json 
from collections import Counter, defaultdict

folder = "../inference/output/GLM-4.6/browsecomp/20251105-214811/"
reflection_file = f"{folder}/iter1.evolved_kflow.jsonl"

it1_judgement = get_iteration_judgement(f"{folder}/iter1_scored.jsonl")
it2_judgement = get_iteration_judgement(f"{folder}/iter2_scored.jsonl")
it3_judgement = get_iteration_judgement(f"{folder}/iter3_scored.jsonl")

# Map iteration number to ground truth
iteration_ground_truth = {1: it1_judgement, 2: it2_judgement, 3: it3_judgement}

# Track overall metrics per iteration
overall_metrics = {1: {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0},
                   2: {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0},
                   3: {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0}}

# Print header
print("qid,iteration_id,GT,correct_count,not_correct_count,category")

with open(reflection_file,'r') as f:
    
    for question_num, line in enumerate(f, start=0):
        data = json.loads(line)
        
        if "history" not in data:
            continue
        
        # For each iteration
        for history_idx, history in enumerate(data["history"]):
            iteration = history.get("iteration", history_idx + 1)
            
            if iteration not in iteration_ground_truth:
                continue
                
            ground_truth = iteration_ground_truth[iteration][question_num]
            
            # Collect all reflector judgements for this iteration
            reflector_judgements = []
            if "reflection_output" in history and history["reflection_output"]:
                for reflection in history["reflection_output"]:
                    judgement = reflection.get("correctness_judgement", "unknown")
                    reflector_judgements.append(judgement)
            
            # Count correct vs not correct
            if reflector_judgements:
                judgement_counts = Counter(reflector_judgements)
                correct_count = judgement_counts.get("correct", 0)
                not_correct_count = sum(v for k, v in judgement_counts.items() if k != "correct")
                total_count = len(reflector_judgements)
                
                # Only correct if ALL judgements are "correct"
                reflector_verdict = "correct" if (correct_count == total_count) else "incorrect"
            else:
                correct_count = 0
                not_correct_count = 0
                reflector_verdict = "unknown"
            
            # Calculate category (TP, FP, TN, FN)
            if ground_truth == "correct" and reflector_verdict == "correct":
                category = "TP"
                overall_metrics[iteration]['TP'] += 1
            elif ground_truth == "incorrect" and reflector_verdict == "correct":
                category = "FP"
                overall_metrics[iteration]['FP'] += 1
            elif ground_truth == "incorrect" and reflector_verdict == "incorrect":
                category = "TN"
                overall_metrics[iteration]['TN'] += 1
            elif ground_truth == "correct" and reflector_verdict == "incorrect":
                category = "FN"
                overall_metrics[iteration]['FN'] += 1
            else:
                category = "UNKNOWN"
            
            # Print CSV-style row
            print(f"{question_num},{iteration},{ground_truth},{correct_count},{not_correct_count},{category}")

# Print summary
print(f"\n{'='*60}")
print("SUMMARY PER ITERATION")
print('='*60)
for iteration in sorted(overall_metrics.keys()):
    metrics = overall_metrics[iteration]
    total = sum(metrics.values())
    print(f"\nIteration {iteration}:")
    print(f"  TP: {metrics['TP']:3d}, FP: {metrics['FP']:3d}, TN: {metrics['TN']:3d}, FN: {metrics['FN']:3d}")
    if total > 0:
        accuracy = (metrics['TP'] + metrics['TN']) / total * 100
        print(f"  Accuracy: {accuracy:.1f}%")

qid,iteration_id,GT,correct_count,not_correct_count,category
0,1,correct,8,0,TP
0,2,correct,8,0,TP
0,3,correct,8,0,TP
1,1,correct,6,2,FN
1,2,correct,0,8,FN
1,3,correct,7,1,FN
2,1,correct,0,8,FN
2,2,incorrect,0,8,TN
2,3,correct,0,8,FN
3,1,incorrect,0,8,TN
3,2,incorrect,8,0,FP
3,3,incorrect,0,8,TN
4,1,correct,8,0,TP
4,2,correct,8,0,TP
4,3,correct,6,2,FN
5,1,incorrect,0,8,TN
5,2,incorrect,0,8,TN
5,3,incorrect,0,8,TN
6,1,correct,7,1,FN
6,2,correct,3,5,FN
6,3,incorrect,0,8,TN
7,1,incorrect,0,8,TN
7,2,incorrect,0,8,TN
7,3,incorrect,0,8,TN
8,1,incorrect,0,8,TN
8,2,correct,0,8,FN
8,3,incorrect,0,8,TN
9,1,incorrect,0,8,TN
9,2,incorrect,0,8,TN
9,3,incorrect,0,8,TN
10,1,incorrect,1,7,TN
10,2,incorrect,0,8,TN
10,3,incorrect,0,8,TN
11,1,correct,7,1,FN
11,2,correct,2,6,FN
11,3,correct,0,8,FN
12,1,incorrect,0,8,TN
12,2,incorrect,0,8,TN
12,3,incorrect,0,8,TN
13,1,incorrect,4,4,TN
13,2,incorrect,0,8,TN
13,3,incorrect,0,8,TN
14,1,incorrect,0,8,TN
14,2,correct,8,0,TP
14,3,incorrect,0,8,TN
15,1,incorrect,